# Import libraries and configure 

In [21]:
from pathlib import Path
from datetime import datetime

import os, json, re
import networkx as nx

from dotenv import load_dotenv
from groq import Groq

RESULTS_DIR = Path("results")
CLUSTER_SUMMARY_PATH = Path("../d1/results/cluster_summary.json")
ALERTS_PATH = Path("dataset/alerts_sample.jsonl")
SERVICES_PATH = Path("dataset/services.json")
HISTORY_PATH = Path("dataset/incidents_history.json")

# Import cluster summary

In [22]:
def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def load_jsonl(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows

cluster_summary = load_json(CLUSTER_SUMMARY_PATH)
alerts_raw = load_jsonl(ALERTS_PATH)
services_raw = load_json(SERVICES_PATH)
incidents_history = load_json(HISTORY_PATH)

print("Cluster_summary:", cluster_summary)
print("Number of alerts:", len(alerts_raw))
print("Number of incidents_history:", len(incidents_history.get("incidents", [])))


Cluster_summary: {'input_alerts': 20, 'output_clusters': 2, 'reduction_ratio': 0.9, 'clusters': [{'cluster_id': 'c-000-000', 'alert_count': 19, 'services': ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'search-svc'], 'time_range': ['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z'], 'max_severity': 'warn', 'fingerprints': ['cart-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'checkout-svc|latency_p99_ms|crit', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'edge-lb|upstream_5xx_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'notification-svc|queue_depth|crit', 'notification-svc|queue_lag_ms|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|db_connection_pool_used_ratio|warn', 'payment-svc|error_rate|crit', 'payment-svc|error_rate|warn', 'payment-svc|latency_p99_ms|crit', 'search-svc|catalog_db_query_time_ms|warn']}, {'cluster_id': 'c-000-001', 'alert_cou

# Build service graph

In [23]:
def build_graph(services_data: str) -> nx.DiGraph:
    g = nx.DiGraph()
    
    # Add service nodes
    for svc in services_data['services']:
        g.add_node(svc['name'], **{k: v for k, v in svc.items() if k != 'name'})
    
    # Add store nodes
    for store in services_data['stores']:
        g.add_node(store['name'], **{k: v for k, v in store.items() if k != 'name'})
    
    # Add edges
    for edge in services_data['edges']:
        g.add_edge(edge['from'], edge['to'], type=edge['type'])
    
    return g



G = build_graph(services_raw)
print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("Sample edges:", list(G.edges())[:10])

Graph nodes: 14
Graph edges: 17
Sample edges: [('edge-lb', 'auth-svc'), ('edge-lb', 'catalog-svc'), ('edge-lb', 'search-svc'), ('edge-lb', 'checkout-svc'), ('checkout-svc', 'cart-svc'), ('checkout-svc', 'payment-svc'), ('checkout-svc', 'inventory-svc'), ('checkout-svc', 'notification-svc'), ('payment-svc', 'payments-db'), ('cart-svc', 'cart-redis')]


# Extract các cluster từ cluster summary

In [24]:
def extract_clusters(cluster_summary):
    clusters = []

    for c in cluster_summary["clusters"]:
        clusters.append({
            "cluster_id": c["cluster_id"],
            "alert_count": c["alert_count"],
            "services": c["services"],
            "time_range": c["time_range"],
            "max_severity": c["max_severity"],
            "fingerprints": c.get("fingerprints", []),
        })

    return clusters


clusters = extract_clusters(cluster_summary)

print("Clusters parsed:", len(clusters))
for c in clusters:
    print(
        c["cluster_id"],
        c["services"],
        c["time_range"],
        "severity=", c["max_severity"],
        "alerts=", c["alert_count"],
        c.get("fingerprints", [])
    )

Clusters parsed: 2
c-000-000 ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'search-svc'] ['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z'] severity= warn alerts= 19 ['cart-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'checkout-svc|latency_p99_ms|crit', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'edge-lb|upstream_5xx_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'notification-svc|queue_depth|crit', 'notification-svc|queue_lag_ms|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|db_connection_pool_used_ratio|warn', 'payment-svc|error_rate|crit', 'payment-svc|error_rate|warn', 'payment-svc|latency_p99_ms|crit', 'search-svc|catalog_db_query_time_ms|warn']
c-000-001 ['recommender-svc'] ['2026-06-12T09:45:10Z', '2026-06-12T09:45:10Z'] severity= warn alerts= 1 ['recommender-svc|cpu_utilization|warn']


# Cho mỗi cluster: chạy graph + temporal scorer → top-K candidates

In [25]:
    
def parse_ts(ts: str):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


def earliest_time_by_service(alerts_raw):

    result = {}

    for alert in alerts_raw:
        service = alert.get("service")
        timestamp = parse_ts(alert.get("ts"))

        if not service or not timestamp:
            continue

        if service not in result or timestamp < result[service]:
            result[service] = timestamp

    return result


EARLIEST = earliest_time_by_service(alerts_raw)

print("Earliest alert time by service:")
for service, t in EARLIEST.items():
    print("- ", service, ": ", t)

Earliest alert time by service:
-  payment-svc :  2026-06-12 09:42:01+00:00
-  checkout-svc :  2026-06-12 09:42:45+00:00
-  edge-lb :  2026-06-12 09:43:15+00:00
-  cart-svc :  2026-06-12 09:43:32+00:00
-  notification-svc :  2026-06-12 09:43:50+00:00
-  recommender-svc :  2026-06-12 09:45:10+00:00
-  search-svc :  2026-06-12 09:46:50+00:00


In [26]:
def temporal_scores(services):

    # Tính điểm cho các service theo thời điểm sớm nhất của alert 
    # Service alert sớm nhất được 1.0.
    # Service alert muộn nhất được 0.0.
    # Nếu thiếu dữ liệu thời gian thì cho 0.5.

    times = {
        service: EARLIEST.get(service)
        for service in services
        if EARLIEST.get(service) is not None
    }

    if len(times) <= 1:
        return {service: 0.5 for service in services}

    min_t = min(times.values())
    max_t = max(times.values())

    span = max((max_t - min_t).total_seconds(), 1.0)

    scores = {}

    for service in services:
        t = times.get(service)

        if t is None:
            scores[service] = 0.5
        else:
            scores[service] = 1.0 - ((t - min_t).total_seconds() / span)
            # print(service, scores[service])

    return scores

In [27]:
def graph_temporal_top_k(cluster, G, top_k=3, graph_weight=0.6, temporal_weight=0.4):

    services = [service for service in cluster["services"] if service]

    if not services:
        return []

    
    # Chỉ lấy subgraph gồm các service đang alert trong cluster
    subgraph = G.subgraph(services).copy()

    # PageRank trên graph A -> B.
    # Nếu checkout-svc -> payment-svc, payment-svc sẽ nhận edge vào nên có PageRank cao hơn.
    if subgraph.number_of_edges() > 0:
        pagerank_scores = nx.pagerank(subgraph, alpha=0.85)
    else:
        pagerank_scores = {
            service: 1 / len(services)
            for service in services
        }

    max_pr = max(pagerank_scores.values()) if pagerank_scores else 1.0

    pagerank_norm = {
        service: pagerank_scores.get(service, 0.0) / max_pr
        for service in services
    }

    time_scores = temporal_scores(services)

    scored = []

    for service in services:
        # terminal node = service không gọi tiếp service nào khác trong cluster
        # Ví dụ: edge-lb -> checkout-svc -> payment-svc
        # payment-svc có out_degree = 0 nên được bonus nhẹ.
        out_degree = subgraph.out_degree(service) if service in subgraph else 0
        terminal_bonus = 0.05 if out_degree == 0 else 0.0

        score = (
            graph_weight * pagerank_norm.get(service, 0.0)
            + temporal_weight * time_scores.get(service, 0.5)
            + terminal_bonus
        )

        score = min(score, 1.0)
        scored.append((service, round(float(score), 4)))

    scored.sort(key=lambda x: x[1], reverse=True)
    
    return scored[:top_k]


In [28]:
for cluster in clusters:
    top_candidates = graph_temporal_top_k(cluster, G, top_k=3)

    print("Cluster:", cluster["cluster_id"])
    print("Top candidates:", top_candidates)
    print()

Cluster: c-000-000
Top candidates: [('payment-svc', 1.0), ('checkout-svc', 0.9391), ('cart-svc', 0.9151)]

Cluster: c-000-001
Top candidates: [('recommender-svc', 0.85)]



# Load incidents_history.json → retrieve top-3 similar (keyword similarity)

In [29]:
CLASS_ENUM = {
    "connection_pool_exhaustion",
    "slow_query",
    "memory_leak",
    "rebalance_storm",
    "deadlock",
    "network_partition",
    "bad_deploy",
    "config_push",
    "tls_expiry",
    "ddos",
    "other"
}


def tokenize_text(text):
    return set(re.findall(r"[a-zA-Z0-9_\-]+", str(text).lower()))


def get_history_list(incidents_history):
    if isinstance(incidents_history, list):
        return incidents_history

    if isinstance(incidents_history, dict):
        return incidents_history.get("incidents", [])

    return []


HISTORY = [
    incident
    for incident in get_history_list(incidents_history)
    if isinstance(incident, dict)
]

print("History incidents loaded:", len(HISTORY))

History incidents loaded: 29


In [30]:
def incident_id(incident):
    return str(incident.get("id", "UNKNOWN"))


def incident_services(incident):
    services = incident.get("services_involved", [])

    if isinstance(services, str):
        services = [services]

    return sorted(set(services))


def incident_root_cause(incident):
    return incident.get("root_cause_service")


def incident_class(incident):
    root_class = incident.get("root_cause_class", "other")

    if root_class not in CLASS_ENUM:
        return "other"

    return root_class


def incident_actions(incident):
    actions = incident.get("remediation", [])

    if isinstance(actions, str):
        actions = [actions]

    if not actions:
        return ["Investigate manually"]

    return actions


def severity_norm(value):
    return str(value or "").strip().lower()

In [31]:
def keyword_similarity(cluster, history_incident):

    # Retrieval indicator: keyword_similarity + kNN-style top_k.
    # oot cause cũ có nằm trong cluster hiện tại thì + 0.4
    # service overlap giữa cluster và incident cũ thì +0.2 cho mỗi lần overlap
    # severity có giống nhau thì + 0.2


    cluster_services = set(cluster["services"])
    history_services = set(incident_services(history_incident))
    history_root = incident_root_cause(history_incident)

    score = 0.0

    # Nếu root cause của incident cũ nằm trong cluster hiện tại
    if history_root in cluster_services:
        score += 0.4

    # Service overlap
    overlap = len(cluster_services & history_services)
    score += min(0.4, 0.2 * overlap)

    # Severity giống nhau
    cluster_severity = severity_norm(cluster.get("severity"))
    history_severity = severity_norm(history_incident.get("severity"))

    if cluster_severity and cluster_severity == history_severity:
        score += 0.2


    return round(min(score, 1.0), 4)

In [32]:
def retrieve_similar(cluster, top_k=3, min_score=0.2):
    
    # Tìm top-K incident history giống cluster hiện tại nhất.

    scored = []

    for history_incident in HISTORY:
        similarity_score = keyword_similarity(cluster, history_incident)

        if similarity_score >= min_score:
            scored.append((history_incident, similarity_score))

    scored.sort(key=lambda x: x[1], reverse=True)

    return scored[:top_k]

In [33]:
for cluster in clusters[:3]:
    similar_incidents = retrieve_similar(cluster, top_k=3)

    print("Cluster:", cluster["cluster_id"])

    for incident, score in similar_incidents:
        print(
            incident_id(incident),
            "score=", score,
            "class=", incident_class(incident),
            "root=", incident_root_cause(incident)
        )

    print()

Cluster: c-000-000
INC-2025-11-08 score= 0.8 class= connection_pool_exhaustion root= payment-svc
INC-2026-03-20 score= 0.8 class= ddos root= edge-lb
INC-2025-08-17 score= 0.6 class= tls_expiry root= edge-lb

Cluster: c-000-001
INC-2025-08-02 score= 0.6 class= memory_leak root= recommender-svc
INC-2025-10-28 score= 0.6 class= other root= recommender-svc
INC-2026-03-07 score= 0.6 class= other root= recommender-svc



# Classifier (required): lấy class + actions từ top-1 similar incident (kNN-style)

In [34]:
def classify_from_top1_similar(cluster):

    # Classifier kNN-style:
    # Retrieve top-3 similar incidents
    # Lấy top-1 incident giống nhất
    # Copy class + actions từ top-1
    # Nếu không tìm được incident tương tự thì fallback


    similar_incidents = retrieve_similar(cluster, top_k=3)

    if not similar_incidents:
        return {
            "class": "other",
            "actions": ["Investigate manually"],
            "similar_incidents": [],
            "retrieval_score": 0.0
        }

    top1_incident, top1_score = similar_incidents[0]

    return {
        "class": incident_class(top1_incident),
        "actions": incident_actions(top1_incident),
        "similar_incidents": [
            incident_id(incident)
            for incident, score in similar_incidents
        ],
        "retrieval_score": top1_score
    }

In [35]:
for cluster in clusters:
    prediction = classify_from_top1_similar(cluster)

    print("Cluster:", cluster["cluster_id"])
    print("Predicted class:", prediction["class"])
    print("Actions:", prediction["actions"])
    print("Similar incidents:", prediction["similar_incidents"])
    print("Retrieval score:", prediction["retrieval_score"])
    print()

Cluster: c-000-000
Predicted class: connection_pool_exhaustion
Actions: ['Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.']
Similar incidents: ['INC-2025-11-08', 'INC-2026-03-20', 'INC-2025-08-17']
Retrieval score: 0.8

Cluster: c-000-001
Predicted class: memory_leak
Actions: ['Patch leak; rollback v3.0 trong khi chờ. Add gc.collect() trong handler.']
Similar incidents: ['INC-2025-08-02', 'INC-2025-10-28', 'INC-2026-03-07']
Retrieval score: 0.6



# Validate output schema + fallback nếu retrieval rỗng

In [36]:

def validate_result(item):
    required = ["cluster_id", "graph_top3", "root_cause", "class", "confidence", "actions", "reasoning", "similar_incidents", "method"]
    for k in required:
        if k not in item:
            return False, f"missing {k}"
    if not isinstance(item["graph_top3"], list) or not item["graph_top3"]:
        return False, "graph_top3 must be non-empty list"
    if item["class"] not in CLASS_ENUM:
        return False, "invalid class"
    if not isinstance(item["confidence"], (int, float)) or not (0 <= item["confidence"] <= 1):
        return False, "invalid confidence"
    if not isinstance(item["actions"], list) or not item["actions"]:
        return False, "actions must be non-empty list"
    return True, "ok"

def analyze_cluster(cluster):
    graph_top3 = graph_temporal_top_k(cluster, G, top_k=3)
    if not graph_top3:
        graph_top3 = [[cluster["services"][0] if cluster["services"] else "unknown", 0.1]]

    root_cause = graph_top3[0][0]
    base_conf = float(graph_top3[0][1])
    similar = retrieve_similar(cluster, top_k=3)

    if similar:
        top_incident, sim_score = similar[0]
        cls = incident_class(top_incident)
        actions = incident_actions(top_incident)
        similar_ids = [incident_id(h) for h, _ in similar]

        confidence = round(base_conf, 4)
        method = "graph+retrieval-knn"

        reasoning = (
            f"Graph+temporal scorer xếp {root_cause} cao nhất trong cluster {cluster['cluster_id']} "
            f"với root-cause confidence theo service graph + temporal là {base_conf:.2f}. "
            f"Retrieval tìm incident tương tự nhất là {incident_id(top_incident)} với similarity {sim_score:.2f}; "
            f"class/actions được lấy từ top-1 similar incident theo kNN-style."
        )
    else:
        cls = "other"
        actions = ["Investigate manually"]
        similar_ids = []
        confidence = round(max(0.1, min(base_conf, 0.65)), 4)
        method = "graph-only-fallback"
        reasoning = (
            f"Không tìm thấy incident lịch sử đủ tương tự cho cluster {cluster['cluster_id']}. "
            f"Fallback dùng top-1 từ graph+temporal là {root_cause}; class để other và yêu cầu điều tra thủ công."
        )

    item = {
        "cluster_id": cluster["cluster_id"],
        "graph_top3": [[s, float(score)] for s, score in graph_top3],
        "root_cause": root_cause,
        "class": cls,
        "confidence": float(confidence),
        "actions": actions,
        "reasoning": reasoning,
        "similar_incidents": similar_ids,
        "method": method,
    }
    ok, msg = validate_result(item)
    if not ok:
        # hard fallback để endpoint/notebook không crash
        item.update({
            "class": "other",
            "confidence": 0.3,
            "actions": ["Investigate manually"],
            "method": "schema-fallback",
            "reasoning": f"Schema invalid ({msg}); fallback to manual investigation."
        })
    return item

results = [analyze_cluster(c) for c in clusters]
rca_output = {"clusters_analyzed": len(results), "results": results}

out_path = RESULTS_DIR / "rca_output.json"
out_path.write_text(json.dumps(rca_output, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", out_path)
print(json.dumps(rca_output, ensure_ascii=False, indent=2)[:2000])


Wrote: results\rca_output.json
{
  "clusters_analyzed": 2,
  "results": [
    {
      "cluster_id": "c-000-000",
      "graph_top3": [
        [
          "payment-svc",
          1.0
        ],
        [
          "checkout-svc",
          0.9391
        ],
        [
          "cart-svc",
          0.9151
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 1.0,
      "actions": [
        "Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%."
      ],
      "reasoning": "Graph+temporal scorer xếp payment-svc cao nhất trong cluster c-000-000 với root-cause confidence theo service graph + temporal là 1.00. Retrieval tìm incident tương tự nhất là INC-2025-11-08 với similarity 0.80; class/actions được lấy từ top-1 similar incident theo kNN-style.",
      "similar_incidents": [
        "INC-2025-11-08",
        "INC-2026-03-20",
        "INC-2025-08-17"
      ],
      "method": "graph+retrieval-knn"
 

# Bonus 3 - LLM enrichment: dùng Groq free tier - thay step 6 bằng LLM call với prompt structure §4.3. 

Compare class label vs kNN top-1. Đọc thêm Anthropic “Building Effective Agents” trước khi làm.

In [37]:
CLASS_ENUM = {
    "connection_pool_exhaustion",
    "slow_query",
    "memory_leak",
    "rebalance_storm",
    "deadlock",
    "network_partition",
    "bad_deploy",
    "config_push",
    "tls_expiry",
    "ddos",
    "other"
}

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)


def build_llm_context(cluster, graph_top3, similar_incidents):
    similar_context = []

    for incident, score in similar_incidents:
        similar_context.append({
            "incident_id": incident_id(incident),
            "similarity_score": score,
            "root_cause_service": incident_root_cause(incident),
            "class": incident_class(incident),
            "actions": incident_actions(incident),
            "services_involved": incident_services(incident),
            "summary": incident.get("summary", ""),
            "description": incident.get("description", ""),
            "severity": incident.get("severity", "")
        })

    return {
        "cluster": {
            "cluster_id": cluster["cluster_id"],
            "services": cluster["services"],
            "severity": cluster.get("severity"),
            "alert_count": cluster.get("alert_count"),
            "fingerprints": cluster.get("fingerprints", []),
            "time_range": cluster.get("time_range", [])
        },
        "graph_top3": graph_top3,
        "similar_incidents": similar_context
    }


def call_groq_rca(cluster, graph_top3, similar_incidents, model="llama-3.1-8b-instant"):

    context = build_llm_context(cluster, graph_top3, similar_incidents)

    system_prompt = """
        You are a senior SRE doing Root Cause Analysis for a microservice incident.

        Return ONLY valid JSON.
        Do not use markdown.
        Do not explain outside JSON.

        Rules:
        - root_cause must be one of the services in the cluster.
        - class must be one of:
        connection_pool_exhaustion, slow_query, memory_leak, rebalance_storm, deadlock,
        network_partition, bad_deploy, config_push, tls_expiry, ddos, other.
        - confidence must be a float between 0 and 1.
        - actions must be a non-empty list of strings.
        - similar_incidents must be a list of incident IDs.
    """

    user_prompt = f"""
        Analyze this RCA context.

        Use graph_top3 to decide root_cause.
        Use similar_incidents to infer class and actions.
        If evidence is weak, lower confidence and use class "other".

        Return JSON with this exact shape:
        {{
            "root_cause": "service-name",
            "class": "connection_pool_exhaustion",
            "confidence": 0.84,
            "actions": ["action 1", "action 2"],
            "reasoning": "short reasoning",
            "similar_incidents": ["INC-xxxx"]
        }}

        Context:
        {json.dumps(context, indent=2, ensure_ascii=False)}
    """

    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        max_tokens=600,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    content = response.choices[0].message.content
    return json.loads(content)

In [38]:
def validate_llm_output(llm_output, cluster):
    services = set(cluster["services"])

    if not isinstance(llm_output, dict):
        return False

    if llm_output.get("root_cause") not in services:
        return False

    if llm_output.get("class") not in CLASS_ENUM:
        return False

    confidence = llm_output.get("confidence")
    if not isinstance(confidence, (int, float)):
        return False

    if not 0 <= float(confidence) <= 1:
        return False

    actions = llm_output.get("actions")
    if not isinstance(actions, list) or len(actions) == 0:
        return False

    return True

In [39]:
def classify_with_groq(cluster, graph_top3):
    similar_incidents = retrieve_similar(cluster, top_k=3)
    knn_prediction = classify_from_top1_similar(cluster)

    if not similar_incidents:
        return {
            "llm_output": None,
            "knn_prediction": knn_prediction,
            "final_prediction": knn_prediction,
            "llm_valid": False,
            "compare": "No similar incidents found, fallback to kNN."
        }

    try:
        llm_output = call_groq_rca(cluster, graph_top3, similar_incidents)

        if validate_llm_output(llm_output, cluster):
            final_prediction = {
                "class": llm_output["class"],
                "actions": llm_output["actions"],
                "similar_incidents": llm_output.get("similar_incidents", knn_prediction["similar_incidents"]),
                "confidence": float(llm_output["confidence"]),
                "reasoning": llm_output["reasoning"],
                "classifier_method": "Groq-LLM-enrichment"
            }

            compare = (
                f"kNN class={knn_prediction['class']} vs "
                f"Groq class={llm_output['class']}"
            )

            return {
                "llm_output": llm_output,
                "knn_prediction": knn_prediction,
                "final_prediction": final_prediction,
                "llm_valid": True,
                "compare": compare
            }

        return {
            "llm_output": llm_output,
            "knn_prediction": knn_prediction,
            "final_prediction": knn_prediction,
            "llm_valid": False,
            "compare": "Groq output invalid, fallback to kNN."
        }

    except Exception as e:
        return {
            "llm_output": None,
            "knn_prediction": knn_prediction,
            "final_prediction": knn_prediction,
            "llm_valid": False,
            "compare": f"Groq call failed: {e}. Fallback to kNN."
        }

In [40]:
for cluster in clusters:
    graph_top3 = graph_temporal_top_k(cluster, G, top_k=3)
    enriched = classify_with_groq(cluster, graph_top3)

    print("Cluster:", cluster["cluster_id"])
    print("Graph top1:", graph_top3[0] if graph_top3 else None)
    print("kNN class:", enriched["knn_prediction"]["class"])
    print("Groq valid:", enriched["llm_valid"])

    if enriched["llm_output"]:
        print("Groq class:", enriched["llm_output"].get("class"))
        print("Groq root cause:", enriched["llm_output"].get("root_cause"))
        print("Groq confidence:", enriched["llm_output"].get("confidence"))

    print("Compare:", enriched["compare"])
    print()

Cluster: c-000-000
Graph top1: ('payment-svc', 1.0)
kNN class: connection_pool_exhaustion
Groq valid: True
Groq class: connection_pool_exhaustion
Groq root cause: payment-svc
Groq confidence: 0.8
Compare: kNN class=connection_pool_exhaustion vs Groq class=connection_pool_exhaustion

Cluster: c-000-001
Graph top1: ('recommender-svc', 0.85)
kNN class: memory_leak
Groq valid: True
Groq class: memory_leak
Groq root cause: recommender-svc
Groq confidence: 0.6
Compare: kNN class=memory_leak vs Groq class=memory_leak

